In [1]:
import numpy as np
import pandas as pd
import sys
from pathlib import Path
sys.path.append('/root/capsule/code/beh_ephys_analysis')
from utils.ephys_functions import fitSpikeModelG
import platform
import os
from pathlib import Path
import shutil
from utils.beh_functions import session_dirs, get_session_tbl, makeSessionDF
from utils.photometry_utils import get_FP_data
from matplotlib import pyplot as plt
from IPython.display import display
from scipy.signal import find_peaks
from harp.clock import align_timestamps_to_anchor_points
import numpy as np
from scipy.signal import butter, filtfilt, medfilt, sosfiltfilt
from scipy.optimize import curve_fit
import json
from sklearn.linear_model import LinearRegression
from matplotlib.gridspec import GridSpec
import pickle
from joblib import Parallel, delayed
import matplotlib.pyplot as plt
import time

# %matplotlib widget
import re
import random
from matplotlib.gridspec import GridSpec
from utils.photometry_combine import population_GLM, plot_tuning_curve, plot_psth, population_GLM_ani
from contextlib import redirect_stdout
%matplotlib inline

In [2]:
session_csv = '/root/capsule/code/data_management/hopkins_FP_session_assets.csv'
session_tbl = pd.read_csv(session_csv)
session_list = session_tbl['session_id'].tolist()
target_folder = '/root/capsule/scratch/manuscript/F_photometry'

In [7]:
region_curr = 'PL'
channel_curr = 'G_tri-exp_mc'
align_curr = 'choice_time'
num_bins = 6
thresh_curr = 0.5
quantiles = False
pre_time = -0.7
post_time = 2.7

params_dict = {
    'region_curr': region_curr,
    'channel_curr': channel_curr,
    'align_curr': align_curr,
    'num_bins': num_bins,
    'thresh_curr': thresh_curr,
    'quantiles': quantiles,
    'pre_time': pre_time,
    'post_time': post_time
}
print(f'Processing tuning curve for region: {region_curr}, channel: {channel_curr}, align: {align_curr}, num_bins: {num_bins}, threshold: {thresh_curr}')

results = plot_tuning_curve(
    session_list, region=region_curr, target_var = 'pe', channel= channel_curr, 
    align=align_curr, pre_time=pre_time, post_time=post_time, num_bins=num_bins, thresh=thresh_curr, quantiles=quantiles
)

save_dir = r'F:\figures\post_python_migration\results'
if not os.path.exists(save_dir):
    os.makedirs(save_dir)
file_name = f'population_tuning_curve_{region_curr}_{channel_curr}_{align_curr}_numbins{num_bins}_thresg{thresh_curr}_quantiles{quantiles}.pkl'
with open(os.path.join(save_dir, file_name), 'wb') as f:
    pickle.dump(results, f)
    pickle.dump(params_dict, f)
print(f'Saved results to {os.path.join(save_dir, file_name)}')
# save fig
results['fig'].savefig(os.path.join(save_dir, file_name.replace('.pkl', '.pdf')), dpi=300)

Processing tuning curve for region: PL, channel: G_tri-exp_mc, align: choice_time, num_bins: 6, threshold: 0.5


/opt/conda/lib/python3.10/site-packages/numpy/_core/fromnumeric.py:3904: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/opt/conda/lib/python3.10/site-packages/numpy/_core/_methods.py:147: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/opt/conda/lib/python3.10/site-packages/numpy/_core/fromnumeric.py:3904: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/opt/conda/lib/python3.10/site-packages/numpy/_core/_methods.py:147: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/opt/conda/lib/python3.10/site-packages/numpy/_core/fromnumeric.py:3904: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/opt/conda/lib/python3.10/site-packages/numpy/_core/_methods.py:147: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/opt/conda/lib/python3

pe not found in trial data for session behavior_672850_2023-07-01_20-36-42.


/opt/conda/lib/python3.10/site-packages/numpy/_core/fromnumeric.py:3904: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/opt/conda/lib/python3.10/site-packages/numpy/_core/_methods.py:147: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/opt/conda/lib/python3.10/site-packages/numpy/_core/fromnumeric.py:3904: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/opt/conda/lib/python3.10/site-packages/numpy/_core/_methods.py:147: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/opt/conda/lib/python3.10/site-packages/numpy/_core/fromnumeric.py:3904: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/opt/conda/lib/python3.10/site-packages/numpy/_core/_methods.py:147: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


pe not found in trial data for session behavior_672850_2023-07-10_18-13-18.


/opt/conda/lib/python3.10/site-packages/numpy/_core/fromnumeric.py:3904: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/opt/conda/lib/python3.10/site-packages/numpy/_core/_methods.py:147: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/opt/conda/lib/python3.10/site-packages/numpy/_core/fromnumeric.py:3904: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/opt/conda/lib/python3.10/site-packages/numpy/_core/_methods.py:147: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/opt/conda/lib/python3.10/site-packages/numpy/_core/fromnumeric.py:3904: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/opt/conda/lib/python3.10/site-packages/numpy/_core/_methods.py:147: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/opt/conda/lib/python3

In [6]:
results['session_list']

('behavior_669489_2023-06-24_14-56-01',
 'behavior_669489_2023-06-26_16-23-02',
 'behavior_669489_2023-06-27_14-20-19',
 'behavior_669489_2023-06-28_13-10-12',
 'behavior_669489_2023-06-29_16-16-02',
 'behavior_669489_2023-06-30_12-51-58',
 'behavior_669489_2023-07-01_16-31-54',
 'behavior_669489_2023-07-02_12-49-06',
 'behavior_669489_2023-07-03_11-31-11',
 'behavior_669489_2023-07-04_14-24-45',
 'behavior_669489_2023-07-05_15-44-55',
 'behavior_669489_2023-07-06_13-50-08',
 'behavior_669489_2023-07-07_15-15-49',
 'behavior_669489_2023-07-08_17-24-38',
 'behavior_669489_2023-07-10_14-24-28',
 'behavior_669489_2023-07-11_16-43-36',
 'behavior_669489_2023-07-12_15-26-19',
 'behavior_669492_2023-06-24_17-28-24',
 'behavior_669492_2023-06-26_19-14-31',
 'behavior_669492_2023-06-29_18-28-17',
 'behavior_669492_2023-06-30_15-51-45',
 'behavior_669492_2023-07-01_19-25-23',
 'behavior_669492_2023-07-02_16-02-53',
 'behavior_669492_2023-07-03_15-55-33',
 'behavior_669492_2023-07-04_17-17-58',
